In [13]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator

In [14]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [15]:
class EvaluationSchema(BaseModel):
    feedback:str=Field(description="Detailed feedback for the essay")
    score:int=Field(description="Score out of 10",le=10,ge=0)

In [16]:
structured_model = model.with_structured_output(EvaluationSchema)

In [17]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [18]:
def evaluate_language(state:UPSCState)->dict[str,list[int]]:
    prompt =f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return{'language_feedback':output.feedback,"individual_scores":[output.score]}

In [19]:
def evaluate_analysis(state:UPSCState)->dict[str,list[int]]:
    prompt =f'Evaluate the depth of the analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return{'analysis_feedback':output.feedback,"individual_scores":[output.score]}

In [20]:
def evaluate_thought(state:UPSCState)->dict[str,list[int]]:
    prompt =f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return{'clarity_feedback':output.feedback,"individual_scores":[output.score]}

In [21]:
def final_evaluate(state:UPSCState)->dict[str,str|int]:
    prompt =f'Based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis - {state['analysis_feedback']} \n clarity of though - {state['clarity_feedback']}'
    overall_feedback=model.invoke(prompt).content
    avg_score=sum(state['individual_scores'])/len(state['individual_scores'])
    return{"overall_feedback":overall_feedback,"avg_score":avg_score}
    

In [23]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_analysis",evaluate_analysis)
graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("final_evaluate",final_evaluate)


graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluate')
graph.add_edge('evaluate_analysis','final_evaluate')
graph.add_edge('evaluate_thought','final_evaluate')

graph.add_edge('final_evaluate',END)

workflow = graph.compile()

In [24]:
initial_state = {
   "essay": """
   # Artificial Intelligence and Large Language Models: Opportunities and Challenges

Artificial Intelligence (AI) is one of the most significant technological developments of the modern age. It refers to machines performing tasks that normally require human intelligence, such as learning, reasoning, decision-making and problem-solving. Large Language Models (LLMs) are an advanced form of AI trained on vast amounts of text data. They can understand and generate human-like language, helping in writing, translation, coding, research and customer support.

For India, AI and LLMs can become powerful tools for inclusive development. In governance, AI-based assistants can help citizens understand government schemes, fill forms and access grievance services in regional languages. This can make public services more accessible, especially for people who are not comfortable with English or complex digital systems.

In education, LLMs can provide personalized learning support to students. They can explain difficult concepts, create practice questions and offer learning material in local languages. Teachers can use AI for lesson planning and identifying learning gaps. In healthcare, AI can assist doctors in diagnosing diseases, analyzing medical reports and spreading health awareness. Similarly, farmers can receive weather updates, crop advice and market information through AI-powered tools.

However, AI also creates serious challenges. LLMs may generate incorrect or misleading information because they predict language patterns rather than truly understand facts. They can also reproduce social biases present in their training data. If used in recruitment, policing or welfare delivery without safeguards, biased AI can lead to discrimination.

Privacy is another major concern. AI systems depend on large volumes of data, including personal information. Misuse of such data can lead to surveillance, profiling and loss of individual freedom. AI can also automate routine jobs, creating employment insecurity for many workers.

Therefore, India must adopt a balanced approach. AI should be transparent, accountable and subject to human oversight. Citizens need digital literacy, workers need reskilling, and AI tools should be available in Indian languages. The goal should not be to replace humans, but to use technology to improve human capability.

In conclusion, AI and LLMs can transform India’s future if guided by ethics, inclusion and constitutional values. Their success will depend not only on technological progress but also on responsible governance.   
   """
}

result = workflow.invoke(initial_state)

print(result)

{'essay': '\n   # Artificial Intelligence and Large Language Models: Opportunities and Challenges\n\nArtificial Intelligence (AI) is one of the most significant technological developments of the modern age. It refers to machines performing tasks that normally require human intelligence, such as learning, reasoning, decision-making and problem-solving. Large Language Models (LLMs) are an advanced form of AI trained on vast amounts of text data. They can understand and generate human-like language, helping in writing, translation, coding, research and customer support.\n\nFor India, AI and LLMs can become powerful tools for inclusive development. In governance, AI-based assistants can help citizens understand government schemes, fill forms and access grievance services in regional languages. This can make public services more accessible, especially for people who are not comfortable with English or complex digital systems.\n\nIn education, LLMs can provide personalized learning support t